# InternVL3_5-1B/

In [ ]:
# Cell 1 - Environment Setup & Verification
# Model:  InternVL3_5-1B/
# visoion model: OpenGVLab/InternViT-6B-448px-V2_5 
# llm backbone:  "_name_or_path": "/root/codespace/checkpoints/Qwen3-0.6B


##   Model type: qwen3
##   Hidden size: 1024
##   Vocab size: 151936
##   Num layers: 28
##   Num heads: 16
#    r=8,
#    lora_alpha=16,
#    lora_dropout=0.05,
#    target_modules=["wqkv", "wo"],
#    bias="none",
#    "force_image_size": 448,
#    "bos_token_id": 151643,
#    "eos_token_id": 151645,
#    "hidden_size": 1024,



import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import transformers
from pathlib import Path






MODEL_DIR = Path("/home/ali/.cache/modelscope/hub/models/OpenGVLab/InternVL3_5-1B/")
TRAIN_JSON = Path("/mnt/share/ali/VLM_Project/hospital_data/train.cleaned.json")
TEST_JSON = Path("/mnt/share/ali/VLM_Project/hospital_data/test.cleaned.json")

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available(), "device:", torch.cuda.current_device(), torch.cuda.get_device_name(0))
print("MODEL_DIR exists:", MODEL_DIR.exists())
print("TRAIN_JSON exists:", TRAIN_JSON.exists())
print("TEST_JSON exists:", TEST_JSON.exists())

In [8]:
import torch, os

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", None))
print("torch.cuda.is_available =", torch.cuda.is_available())
print("torch.cuda.device_count =", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
    print("Current device:", torch.cuda.current_device())


CUDA_VISIBLE_DEVICES = None
torch.cuda.is_available = True
torch.cuda.device_count = 4
GPU 0: NVIDIA H800 PCIe
GPU 1: NVIDIA H800 PCIe
GPU 2: NVIDIA H800 PCIe
GPU 3: NVIDIA H800 PCIe
Current device: 0


In [9]:
# Cell 1A: Single GPU (use GPU 0 by default)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
print("CUDA_VISIBLE_DEVICES set to:", os.environ["CUDA_VISIBLE_DEVICES"])


CUDA_VISIBLE_DEVICES set to: 2


In [10]:
# Cell 1B: Multi-GPU (leave as-is)
import os
print("Keeping CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", None))


Keeping CUDA_VISIBLE_DEVICES = 2


In [11]:
# Cell 2
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_DIR = Path("/home/ali/.cache/modelscope/hub/models/OpenGVLab/InternVL3_5-1B/")
assert MODEL_DIR.exists(), f"MODEL_DIR not found: {MODEL_DIR}"
print("MODEL_DIR:", MODEL_DIR)

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    trust_remote_code=True,
    use_fast=False
)

model = AutoModel.from_pretrained(
    str(MODEL_DIR),
    trust_remote_code=True,
    torch_dtype="auto",
    device_map="auto"   # auto place/shard depending on visible GPUs
)

model.eval()
print("Loaded:", type(model))


MODEL_DIR: /home/ali/.cache/modelscope/hub/models/OpenGVLab/InternVL3_5-1B


`torch_dtype` is deprecated! Use `dtype` instead!


FlashAttention2 is not installed.
Loaded: <class 'transformers_modules.InternVL3_5_hyphen_1B.modeling_internvl_chat.InternVLChatModel'>


In [12]:
# Cell 3
import torch

print("device_count:", torch.cuda.device_count())
p0 = next(model.parameters())
print("First param device:", p0.device, "dtype:", p0.dtype)

# quick memory peek
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / (1024**3)
        reserv = torch.cuda.memory_reserved(i) / (1024**3)
        print(f"GPU {i}: allocated={alloc:.3f} GB, reserved={reserv:.3f} GB")


device_count: 4
First param device: cuda:0 dtype: torch.bfloat16
GPU 0: allocated=0.885 GB, reserved=0.887 GB
GPU 1: allocated=1.091 GB, reserved=1.096 GB
GPU 2: allocated=0.000 GB, reserved=0.000 GB
GPU 3: allocated=0.000 GB, reserved=0.000 GB


In [13]:
# Cell 3
import torch

print("device_count:", torch.cuda.device_count())
p0 = next(model.parameters())
print("First param device:", p0.device, "dtype:", p0.dtype)

# quick memory peek
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        alloc = torch.cuda.memory_allocated(i) / (1024**3)
        reserv = torch.cuda.memory_reserved(i) / (1024**3)
        print(f"GPU {i}: allocated={alloc:.3f} GB, reserved={reserv:.3f} GB")


device_count: 4
First param device: cuda:0 dtype: torch.bfloat16
GPU 0: allocated=0.885 GB, reserved=0.887 GB
GPU 1: allocated=1.091 GB, reserved=1.096 GB
GPU 2: allocated=0.000 GB, reserved=0.000 GB
GPU 3: allocated=0.000 GB, reserved=0.000 GB


In [3]:
# Cell 2
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # for determinism (can reduce speed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42  # <-- set to your existing seed
seed_everything(SEED)
print("Seed set to:", SEED)


Seed set to: 42


In [7]:
# Cell 3
os.environ["TOKENIZERS_PARALLELISM"] = "2"
# If you want to force a specific GPU (example: only GPU 2)
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"

print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES", None))
print("torch.cuda.is_available =", torch.cuda.is_available())


CUDA_VISIBLE_DEVICES = None
torch.cuda.is_available = True


In [2]:
# Cell 2 — Check bitsandbytes + load InternVL2.5 in 4-bit

import importlib
bnb_spec = importlib.util.find_spec("bitsandbytes")
print("bitsandbytes installed:", bnb_spec is not None)

import torch
from transformers import AutoTokenizer, AutoModel
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

model = AutoModel.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="cuda:0",
)

model.eval()

print("Loaded model:", type(model))
print("img_context_token_id:", getattr(model, "img_context_token_id", None))
print("language_model type:", type(model.language_model))

# quick memory check
print("cuda mem allocated (GB):", torch.cuda.memory_allocated()/1024**3)
print("cuda mem reserved  (GB):", torch.cuda.memory_reserved()/1024**3)


bitsandbytes installed: True


The tokenizer you are loading from '/home/ali/.cache/modelscope/hub/models/OpenGVLab/InternVL3_5-1B' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


FlashAttention2 is not installed.
Loaded model: <class 'transformers_modules.InternVL3_5_hyphen_1B.modeling_internvl_chat.InternVLChatModel'>
img_context_token_id: None
language_model type: <class 'transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM'>
cuda mem allocated (GB): 0.9433126449584961
cuda mem reserved  (GB): 0.986328125


In [3]:
# Cell 3 — Recover img_context_token_id and set it on the model

# Try common candidates
candidates = ["<IMG_CONTEXT>", "<img_context>", "<image_context>", "<im_context>", "<IMG_CONTEXT_TOKEN>"]
found = []

for tok in candidates:
    tid = tokenizer.convert_tokens_to_ids(tok)
    if tid is not None and tid != tokenizer.unk_token_id:
        found.append((tok, tid))

print("Tokenizer unk_token_id:", tokenizer.unk_token_id)
print("Found candidate tokens:", found)

# Fallback: use the known id from your previous full-precision run if nothing found
if len(found) == 0:
    img_ctx_id = 92546
    print("⚠️ No token string found; falling back to known img_context_token_id =", img_ctx_id)
else:
    # pick the first match
    img_ctx_id = found[0][1]
    print("✅ Using img_context_token_id from tokenizer:", img_ctx_id)

# set on model
model.img_context_token_id = img_ctx_id
print("model.img_context_token_id set to:", model.img_context_token_id)


Tokenizer unk_token_id: None
Found candidate tokens: [('<IMG_CONTEXT>', 151671)]
✅ Using img_context_token_id from tokenizer: 151671
model.img_context_token_id set to: 151671


In [ ]:
# Cell 4 — Load cleaned data + filter tags + subject-wise split + label maps

import json, random
from collections import Counter, defaultdict
from pathlib import Path

ALLOWED_TAGS = {"step_classification", "stage_classification"}
SEED = 42
random.seed(SEED)

def load_json(p: Path):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def get_subject(s):
    meta = s.get("meta", {}) or {}
    if meta.get("subject"):
        return meta["subject"]
    _id = s.get("id", "")
    return _id.split("__")[0] if "__" in _id else "UNKNOWN"

def get_label(s):
    meta = s.get("meta", {}) or {}
    if s["main_tag"] == "step_classification":
        return meta.get("step_id")
    else:
        return meta.get("stage_id")

train_all = load_json(TRAIN_JSON)
test_all  = load_json(TEST_JSON)

train_kept = [s for s in train_all if s.get("main_tag") in ALLOWED_TAGS]
test_kept  = [s for s in test_all  if s.get("main_tag") in ALLOWED_TAGS]

print("train_kept:", len(train_kept), Counter([s["main_tag"] for s in train_kept]))
print("test_kept :", len(test_kept),  Counter([s["main_tag"] for s in test_kept]))

# subject split from train_kept
idx_by_subj = defaultdict(list)
for i, s in enumerate(train_kept):
    idx_by_subj[get_subject(s)].append(i)

subjects = sorted(idx_by_subj.keys())
rng = random.Random(SEED)
rng.shuffle(subjects)

VAL_SUBJECT_RATIO = 0.2
n_val_subj = max(1, int(round(len(subjects) * VAL_SUBJECT_RATIO)))
val_subjects = set(subjects[:n_val_subj])

train_idx, val_idx = [], []
for subj, idxs in idx_by_subj.items():
    (val_idx if subj in val_subjects else train_idx).extend(idxs)

train_split = [train_kept[i] for i in train_idx]
val_split   = [train_kept[i] for i in val_idx]

print("train_split:", len(train_split), Counter([s["main_tag"] for s in train_split]))
print("val_split  :", len(val_split),   Counter([s["main_tag"] for s in val_split]))
print("val subjects:", sorted(list(val_subjects))[:20])

# label lists from full train_kept
label_list = {
    "step_classification": sorted({get_label(s) for s in train_kept if s["main_tag"]=="step_classification"}),
    "stage_classification": sorted({get_label(s) for s in train_kept if s["main_tag"]=="stage_classification"}),
}
label2id = {t: {lab:i for i, lab in enumerate(label_list[t])} for t in label_list}

print("step classes :", len(label_list["step_classification"]), label_list["step_classification"])
print("stage classes:", len(label_list["stage_classification"]), label_list["stage_classification"])


In [ ]:
# Cell 5 — QLoRA attach + dataset + 1 epoch finetune + val metrics

import torch, types
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score

from peft import LoraConfig, get_peft_model, TaskType

device = torch.device("cuda:0")

# ---- attach LoRA to language model (4-bit base) ----
lm = model.language_model
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["wqkv", "wo"],
    bias="none",
)
lm = get_peft_model(lm, lora_cfg)
model.language_model = lm

# make sure generate-related cache patch doesn't break anything (safe no-op for training)
if not hasattr(model.language_model, "_orig_prepare_inputs_for_generation"):
    model.language_model._orig_prepare_inputs_for_generation = model.language_model.prepare_inputs_for_generation

def safe_prepare_inputs_for_generation(self, input_ids, past_key_values=None, attention_mask=None, inputs_embeds=None, **kwargs):
    bad_cache = False
    try:
        if past_key_values is not None:
            if len(past_key_values) == 0:
                bad_cache = True
            else:
                first = past_key_values[0]
                if first is None:
                    bad_cache = True
                elif isinstance(first, (tuple, list)) and len(first) > 0 and first[0] is None:
                    bad_cache = True
    except Exception:
        bad_cache = True
    if bad_cache:
        past_key_values = None
    return self._orig_prepare_inputs_for_generation(
        input_ids=input_ids,
        past_key_values=past_key_values,
        attention_mask=attention_mask,
        inputs_embeds=inputs_embeds,
        **kwargs
    )

model.language_model.prepare_inputs_for_generation = types.MethodType(
    safe_prepare_inputs_for_generation, model.language_model
)

# ---- head ----
n_step  = len(label_list["step_classification"])
n_stage = len(label_list["stage_classification"])

class MultiTaskHead(nn.Module):
    def __init__(self, hidden=4096, n_step=13, n_stage=3):
        super().__init__()
        self.step = nn.Linear(hidden, n_step)
        self.stage = nn.Linear(hidden, n_stage)
    def forward(self, x):
        return {"step_logits": self.step(x), "stage_logits": self.stage(x)}

head = MultiTaskHead(hidden=4096, n_step=n_step, n_stage=n_stage).to(device)

# ---- dataset (short prompts + img_ctx tokens) ----
IMG_SIZE = 448
N_FRAMES = 8
CTX_PER_IMAGE = 256
IMG_CTX_ID = model.img_context_token_id

img_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406),
                         std=(0.229, 0.224, 0.225)),
])

def build_inputs(n_frames: int, tag: str):
    bos = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.cls_token_id
    eos = tokenizer.eos_token_id
    n_ctx = n_frames * CTX_PER_IMAGE

    prompt = "Classify step. Answer step_XXX." if tag == "step_classification" else "Classify stage. Answer stage_XX."
    prompt_ids = tokenizer(prompt, add_special_tokens=False).input_ids

    ids = [bos] + [IMG_CTX_ID] * n_ctx + prompt_ids + ([eos] if eos is not None else [])
    input_ids = torch.tensor(ids, dtype=torch.long)
    attention_mask = torch.ones_like(input_ids, dtype=torch.long)
    image_flags = torch.ones((n_frames,), dtype=torch.long)
    return input_ids, attention_mask, image_flags

class SurgicalDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        tag = s["main_tag"]
        meta = s.get("meta", {}) or {}
        y = label2id[tag][meta["step_id"] if tag=="step_classification" else meta["stage_id"]]

        frames = s["frames"][:N_FRAMES]
        imgs = [Image.open(p).convert("RGB") for p in frames]
        pixel_values = torch.stack([img_transform(im) for im in imgs], dim=0)  # [N,3,H,W]

        input_ids, attention_mask, image_flags = build_inputs(pixel_values.shape[0], tag)
        return {"pixel_values": pixel_values,
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "image_flags": image_flags,
                "tag": tag,
                "y": y}

def collate_fn(batch):
    max_len = max(b["input_ids"].shape[0] for b in batch)
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

    def pad(x, v):
        if x.shape[0] == max_len:
            return x
        return torch.cat([x, torch.full((max_len-x.shape[0],), v, dtype=x.dtype)], dim=0)

    pixel_values = torch.stack([b["pixel_values"] for b in batch], dim=0)  # [B,N,3,H,W]
    input_ids = torch.stack([pad(b["input_ids"], pad_id) for b in batch], dim=0)
    attention_mask = torch.stack([pad(b["attention_mask"], 0) for b in batch], dim=0)
    image_flags = torch.stack([b["image_flags"] for b in batch], dim=0)  # [B,N]
    tags = [b["tag"] for b in batch]
    y = torch.tensor([b["y"] for b in batch], dtype=torch.long)
    return {"pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "image_flags": image_flags,
            "tags": tags,
            "y": y}

train_ds = SurgicalDataset(train_split)
val_ds   = SurgicalDataset(val_split)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

# ---- optimizer (LoRA + head) ----
params = [p for p in list(model.language_model.parameters()) + list(head.parameters()) if p.requires_grad]
optim = torch.optim.AdamW(params, lr=2e-4, weight_decay=0.01)

# ---- train 1 epoch with grad accumulation ----
ACCUM = 8
model.train(); head.train()
optim.zero_grad(set_to_none=True)

running = 0.0
for i, batch in enumerate(tqdm(train_loader, desc="qlora finetune(1 epoch)"), start=1):
    B, N, C, H, W = batch["pixel_values"].shape
    pixel_values_bn = batch["pixel_values"].to(device=device, dtype=torch.bfloat16).view(B*N, C, H, W)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    image_flags = batch["image_flags"].to(device).view(B*N)
    tags = batch["tags"]
    y = batch["y"].to(device)

    out = model(
        pixel_values=pixel_values_bn,
        input_ids=input_ids,
        attention_mask=attention_mask,
        image_flags=image_flags,
        output_hidden_states=True,
        return_dict=True,
        use_cache=False,
    )
    pooled = out.hidden_states[-1][:, -1, :].float()
    lg = head(pooled)

    if tags[0] == "step_classification":
        loss = F.cross_entropy(lg["step_logits"], y)
    else:
        loss = F.cross_entropy(lg["stage_logits"], y)

    (loss / ACCUM).backward()

    if i % ACCUM == 0:
        optim.step()
        optim.zero_grad(set_to_none=True)

    running += float(loss.item())

print("Train avg loss:", running / len(train_ds))

# ---- eval ----
model.eval(); head.eval()
ys_step, ps_step, ys_stage, ps_stage = [], [], [], []

with torch.no_grad():
    for batch in val_loader:
        B, N, C, H, W = batch["pixel_values"].shape
        pixel_values_bn = batch["pixel_values"].to(device=device, dtype=torch.bfloat16).view(B*N, C, H, W)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        image_flags = batch["image_flags"].to(device).view(B*N)
        tag = batch["tags"][0]
        y = int(batch["y"].item())

        out = model(
            pixel_values=pixel_values_bn,
            input_ids=input_ids,
            attention_mask=attention_mask,
            image_flags=image_flags,
            output_hidden_states=True,
            return_dict=True,
            use_cache=False,
        )
        pooled = out.hidden_states[-1][:, -1, :].float()
        lg = head(pooled)

        if tag == "step_classification":
            p = int(torch.argmax(lg["step_logits"], dim=-1).item())
            ys_step.append(y); ps_step.append(p)
        else:
            p = int(torch.argmax(lg["stage_logits"], dim=-1).item())
            ys_stage.append(y); ps_stage.append(p)

def summarize(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "n": len(y_true),
    }

print("VAL step :", summarize(ys_step, ps_step))
print("VAL stage:", summarize(ys_stage, ps_stage))

# memory info
print("cuda mem allocated (GB):", torch.cuda.memory_allocated()/1024**3)
print("cuda mem reserved  (GB):", torch.cuda.memory_reserved()/1024**3)


In [6]:
# Cell 6 — Fix vision dtype mismatch (use fp16 pixel_values) + quick single forward

import torch

device = torch.device("cuda:0")

# Inspect vision patch conv dtypes
patch = model.vision_model.embeddings.patch_embedding
print("patch weight dtype:", patch.weight.dtype)
print("patch bias dtype  :", patch.bias.dtype if patch.bias is not None else None)

# Take one batch and test forward with fp16 pixel_values
batch = next(iter(train_loader))
B, N, C, H, W = batch["pixel_values"].shape

pixel_values_bn = batch["pixel_values"].to(device=device, dtype=torch.float16).view(B*N, C, H, W)  # fp16 here
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
image_flags = batch["image_flags"].to(device).view(B*N)

with torch.no_grad():
    out = model(
        pixel_values=pixel_values_bn,
        input_ids=input_ids,
        attention_mask=attention_mask,
        image_flags=image_flags,
        output_hidden_states=True,
        return_dict=True,
        use_cache=False,
    )

print("Forward OK. last hidden:", out.hidden_states[-1].shape, out.hidden_states[-1].dtype)


patch weight dtype: torch.float16
patch bias dtype  : torch.float16


NameError: name 'train_loader' is not defined

In [7]:
# Cell 7 — QLoRA finetune (1 epoch) with fp16 pixel_values + val metrics

model.train(); head.train()
optim.zero_grad(set_to_none=True)

ACCUM = 8
running = 0.0

for i, batch in enumerate(tqdm(train_loader, desc="qlora finetune(1 epoch)"), start=1):
    B, N, C, H, W = batch["pixel_values"].shape

    # IMPORTANT: fp16 for vision
    pixel_values_bn = batch["pixel_values"].to(device=device, dtype=torch.float16).view(B*N, C, H, W)

    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    image_flags = batch["image_flags"].to(device).view(B*N)
    tags = batch["tags"]
    y = batch["y"].to(device)

    out = model(
        pixel_values=pixel_values_bn,
        input_ids=input_ids,
        attention_mask=attention_mask,
        image_flags=image_flags,
        output_hidden_states=True,
        return_dict=True,
        use_cache=False,
    )
    pooled = out.hidden_states[-1][:, -1, :].float()
    lg = head(pooled)

    if tags[0] == "step_classification":
        loss = F.cross_entropy(lg["step_logits"], y)
    else:
        loss = F.cross_entropy(lg["stage_logits"], y)

    (loss / ACCUM).backward()

    if i % ACCUM == 0:
        optim.step()
        optim.zero_grad(set_to_none=True)

    running += float(loss.item())

print("Train avg loss:", running / len(train_ds))

# ---- eval ----
model.eval(); head.eval()
ys_step, ps_step, ys_stage, ps_stage = [], [], [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="eval(val)"):
        B, N, C, H, W = batch["pixel_values"].shape
        pixel_values_bn = batch["pixel_values"].to(device=device, dtype=torch.float16).view(B*N, C, H, W)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        image_flags = batch["image_flags"].to(device).view(B*N)
        tag = batch["tags"][0]
        ytrue = int(batch["y"].item())

        out = model(
            pixel_values=pixel_values_bn,
            input_ids=input_ids,
            attention_mask=attention_mask,
            image_flags=image_flags,
            output_hidden_states=True,
            return_dict=True,
            use_cache=False,
        )
        pooled = out.hidden_states[-1][:, -1, :].float()
        lg = head(pooled)

        if tag == "step_classification":
            pred = int(torch.argmax(lg["step_logits"], dim=-1).item())
            ys_step.append(ytrue); ps_step.append(pred)
        else:
            pred = int(torch.argmax(lg["stage_logits"], dim=-1).item())
            ys_stage.append(ytrue); ps_stage.append(pred)

def summarize(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "n": len(y_true),
    }

print("VAL step :", summarize(ys_step, ps_step))
print("VAL stage:", summarize(ys_stage, ps_stage))

print("cuda mem allocated (GB):", torch.cuda.memory_allocated()/1024**3)
print("cuda mem reserved  (GB):", torch.cuda.memory_reserved()/1024**3)

model.train(); head.train()


qlora finetune(1 epoch):   0%|          | 0/598 [00:00<?, ?it/s]

Train avg loss: 2.3442647229151232


eval(val):   0%|          | 0/160 [00:00<?, ?it/s]

VAL step : {'acc': 0.0875, 'macro_f1': 0.01282051282051282, 'kappa': 0.0005134348793429666, 'n': 80}
VAL stage: {'acc': 0.5125, 'macro_f1': 0.2258953168044077, 'kappa': 0.0, 'n': 80}
cuda mem allocated (GB): 6.832579612731934
cuda mem reserved  (GB): 45.68359375


MultiTaskHead(
  (step): Linear(in_features=4096, out_features=13, bias=True)
  (stage): Linear(in_features=4096, out_features=3, bias=True)
)

In [8]:
# Cell 8 — Pool at last IMG_CTX token position (visual-aligned pooling)

batch = next(iter(train_loader))
B, N, C, H, W = batch["pixel_values"].shape

pixel_values_bn = batch["pixel_values"].to(device=device, dtype=torch.float16).view(B*N, C, H, W)
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
image_flags = batch["image_flags"].to(device).view(B*N)

# position of the last IMG_CTX token in our constructed input
n_ctx = N * CTX_PER_IMAGE
img_ctx_pos = 1 + n_ctx - 1   # after BOS, last context token index
print("seq_len:", input_ids.shape[1], "n_ctx:", n_ctx, "img_ctx_pos:", img_ctx_pos)
print("check token at img_ctx_pos == IMG_CTX_ID:", int(input_ids[0, img_ctx_pos].item()) == IMG_CTX_ID)

model.eval(); head.eval()
with torch.no_grad():
    out = model(
        pixel_values=pixel_values_bn,
        input_ids=input_ids,
        attention_mask=attention_mask,
        image_flags=image_flags,
        output_hidden_states=True,
        return_dict=True,
        use_cache=False,
    )
hs = out.hidden_states[-1]           # [B, seq, 4096]
pooled_img = hs[:, img_ctx_pos, :].float()   # visual-aligned pooled
pooled_last = hs[:, -1, :].float()           # previous pooled

lg_img = head(pooled_img)
lg_last = head(pooled_last)

print("pooled_img:", pooled_img.shape, pooled_img.mean().item(), pooled_img.std().item())
print("pooled_last:", pooled_last.shape, pooled_last.mean().item(), pooled_last.std().item())
print("logits(step) img/last:", lg_img["step_logits"].shape, lg_last["step_logits"].shape)
print("logits(stage) img/last:", lg_img["stage_logits"].shape, lg_last["stage_logits"].shape)


seq_len: 2059 n_ctx: 2048 img_ctx_pos: 2048
check token at img_ctx_pos == IMG_CTX_ID: True
pooled_img: torch.Size([1, 4096]) 0.01940350979566574 2.4611830711364746
pooled_last: torch.Size([1, 4096]) -0.018418608233332634 1.8934435844421387
logits(step) img/last: torch.Size([1, 13]) torch.Size([1, 13])
logits(stage) img/last: torch.Size([1, 3]) torch.Size([1, 3])


In [9]:
# Cell 9 — Retrain 1 epoch using IMG_CTX pooling + val metrics

model.train(); head.train()
optim.zero_grad(set_to_none=True)

ACCUM = 8
running = 0.0

for i, batch in enumerate(tqdm(train_loader, desc="qlora finetune(1 epoch, img_ctx_pool)"), start=1):
    B, N, C, H, W = batch["pixel_values"].shape
    pixel_values_bn = batch["pixel_values"].to(device=device, dtype=torch.float16).view(B*N, C, H, W)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    image_flags = batch["image_flags"].to(device).view(B*N)
    tags = batch["tags"]
    y = batch["y"].to(device)

    n_ctx = N * CTX_PER_IMAGE
    img_ctx_pos = 1 + n_ctx - 1

    out = model(
        pixel_values=pixel_values_bn,
        input_ids=input_ids,
        attention_mask=attention_mask,
        image_flags=image_flags,
        output_hidden_states=True,
        return_dict=True,
        use_cache=False,
    )
    hs = out.hidden_states[-1]
    pooled = hs[:, img_ctx_pos, :].float()   # <-- KEY CHANGE

    lg = head(pooled)

    if tags[0] == "step_classification":
        loss = F.cross_entropy(lg["step_logits"], y)
    else:
        loss = F.cross_entropy(lg["stage_logits"], y)

    (loss / ACCUM).backward()

    if i % ACCUM == 0:
        optim.step()
        optim.zero_grad(set_to_none=True)

    running += float(loss.item())

print("Train avg loss:", running / len(train_ds))

# ---- eval ----
model.eval(); head.eval()
ys_step, ps_step, ys_stage, ps_stage = [], [], [], []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="eval(val,img_ctx_pool)"):
        B, N, C, H, W = batch["pixel_values"].shape
        pixel_values_bn = batch["pixel_values"].to(device=device, dtype=torch.float16).view(B*N, C, H, W)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        image_flags = batch["image_flags"].to(device).view(B*N)
        tag = batch["tags"][0]
        ytrue = int(batch["y"].item())

        n_ctx = N * CTX_PER_IMAGE
        img_ctx_pos = 1 + n_ctx - 1

        out = model(
            pixel_values=pixel_values_bn,
            input_ids=input_ids,
            attention_mask=attention_mask,
            image_flags=image_flags,
            output_hidden_states=True,
            return_dict=True,
            use_cache=False,
        )
        hs = out.hidden_states[-1]
        pooled = hs[:, img_ctx_pos, :].float()

        lg = head(pooled)

        if tag == "step_classification":
            pred = int(torch.argmax(lg["step_logits"], dim=-1).item())
            ys_step.append(ytrue); ps_step.append(pred)
        else:
            pred = int(torch.argmax(lg["stage_logits"], dim=-1).item())
            ys_stage.append(ytrue); ps_stage.append(pred)

print("VAL step :", summarize(ys_step, ps_step))
print("VAL stage:", summarize(ys_stage, ps_stage))

model.train(); head.train()


qlora finetune(1 epoch, img_ctx_pool):   0%|          | 0/598 [00:00<?, ?it/s]

/home/ali/.local/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Train avg loss: 2.0152776831119357


eval(val,img_ctx_pool):   0%|          | 0/160 [00:00<?, ?it/s]

VAL step : {'acc': 0.0875, 'macro_f1': 0.04414747211357381, 'kappa': 0.011677102724657495, 'n': 80}
VAL stage: {'acc': 0.5125, 'macro_f1': 0.2258953168044077, 'kappa': 0.0, 'n': 80}


MultiTaskHead(
  (step): Linear(in_features=4096, out_features=13, bias=True)
  (stage): Linear(in_features=4096, out_features=3, bias=True)
)

In [10]:
# Cell 10 — Run 20 epochs QLoRA baseline (IMG_CTX pooling) + best checkpoint + final TEST

import copy, time
from tqdm.auto import tqdm

EPOCHS = 20
ACCUM = 8

def forward_pooled(batch):
    B, N, C, H, W = batch["pixel_values"].shape
    pixel_values_bn = batch["pixel_values"].to(device=device, dtype=torch.float16).view(B*N, C, H, W)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    image_flags = batch["image_flags"].to(device).view(B*N)

    n_ctx = N * CTX_PER_IMAGE
    img_ctx_pos = 1 + n_ctx - 1

    out = model(
        pixel_values=pixel_values_bn,
        input_ids=input_ids,
        attention_mask=attention_mask,
        image_flags=image_flags,
        output_hidden_states=True,
        return_dict=True,
        use_cache=False,
    )
    hs = out.hidden_states[-1]
    pooled = hs[:, img_ctx_pos, :].float()
    return pooled, batch["tags"], batch["y"].to(device)

def eval_loader(loader, desc="eval"):
    model.eval(); head.eval()
    ys_step, ps_step, ys_stage, ps_stage = [], [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            pooled, tags, y = forward_pooled(batch)
            lg = head(pooled)
            tag = tags[0]
            ytrue = int(y.item())
            if tag == "step_classification":
                pred = int(torch.argmax(lg["step_logits"], dim=-1).item())
                ys_step.append(ytrue); ps_step.append(pred)
            else:
                pred = int(torch.argmax(lg["stage_logits"], dim=-1).item())
                ys_stage.append(ytrue); ps_stage.append(pred)
    return summarize(ys_step, ps_step), summarize(ys_stage, ps_stage)

best = {"score": -1.0, "epoch": 0, "head": None, "lora": None}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train(); head.train()
    optim.zero_grad(set_to_none=True)

    running = 0.0
    for i, batch in enumerate(tqdm(train_loader, desc=f"train epoch {epoch}/{EPOCHS}"), start=1):
        pooled, tags, y = forward_pooled(batch)
        lg = head(pooled)

        if tags[0] == "step_classification":
            loss = F.cross_entropy(lg["step_logits"], y)
        else:
            loss = F.cross_entropy(lg["stage_logits"], y)

        (loss / ACCUM).backward()

        if i % ACCUM == 0:
            optim.step()
            optim.zero_grad(set_to_none=True)

        running += float(loss.item())

    train_loss = running / len(train_ds)

    # val
    step_res, stage_res = eval_loader(val_loader, desc=f"val epoch {epoch}")

    # pick your model selection metric (baseline-friendly):
    # average macro-F1 across the two tasks
    score = 0.5 * (step_res["macro_f1"] + stage_res["macro_f1"])

    dt = time.time() - t0
    print(f"\nEpoch {epoch:02d} | {dt/60:.1f} min | train_loss {train_loss:.4f} | "
          f"VAL step acc {step_res['acc']:.3f} f1 {step_res['macro_f1']:.3f} k {step_res['kappa']:.3f} | "
          f"VAL stage acc {stage_res['acc']:.3f} f1 {stage_res['macro_f1']:.3f} k {stage_res['kappa']:.3f} | "
          f"score {score:.3f}")

    # save best
    if score > best["score"]:
        best["score"] = score
        best["epoch"] = epoch
        best["head"] = copy.deepcopy(head.state_dict())
        # save LoRA adapter weights only (peft)
        best["lora"] = copy.deepcopy(model.language_model.state_dict())
        print(f"✅ New best @ epoch {epoch} (score={score:.3f})")

print("\n=== Best epoch ===", best["epoch"], "best score:", best["score"])

# ---- Build TEST loader and evaluate using best checkpoint ----
# Restore best
head.load_state_dict(best["head"])
model.language_model.load_state_dict(best["lora"], strict=False)

test_ds = SurgicalDataset(test_kept)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_fn)

test_step, test_stage = eval_loader(test_loader, desc="TEST(best)")
print("\nTEST step :", test_step)
print("TEST stage:", test_stage)


train epoch 1/20:   0%|          | 0/598 [00:00<?, ?it/s]

/home/ali/.local/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


val epoch 1:   0%|          | 0/160 [00:00<?, ?it/s]


Epoch 01 | 11.2 min | train_loss 1.8421 | VAL step acc 0.113 f1 0.078 k 0.034 | VAL stage acc 0.625 f1 0.415 k 0.263 | score 0.247
✅ New best @ epoch 1 (score=0.247)


train epoch 2/20:   0%|          | 0/598 [00:00<?, ?it/s]

...

/home/ali/.local/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


val epoch 20:   0%|          | 0/160 [00:00<?, ?it/s]


Epoch 20 | 22.2 min | train_loss 0.0001 | VAL step acc 0.838 f1 0.819 k 0.823 | VAL stage acc 0.950 f1 0.924 k 0.917 | score 0.872

=== Best epoch === 15 best score: 0.8771505860780408


TEST(best):   0%|          | 0/224 [00:00<?, ?it/s]


TEST step : {'acc': 0.7857142857142857, 'macro_f1': 0.785767302872566, 'kappa': 0.7676950998185118, 'n': 112}
TEST stage: {'acc': 0.9553571428571429, 'macro_f1': 0.9527818849676849, 'kappa': 0.9294265910523, 'n': 112}


In [12]:
# Class-wise F1 on TEST (step + stage)

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# --- run predictions on test_loader ---
model.eval(); head.eval()

ys_step, ps_step = [], []
ys_stage, ps_stage = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="predict(TEST)"):
        pooled, tags, y = forward_pooled(batch)  # uses IMG_CTX pooling
        lg = head(pooled)
        tag = tags[0]
        ytrue = int(y.item())

        if tag == "step_classification":
            pred = int(torch.argmax(lg["step_logits"], dim=-1).item())
            ys_step.append(ytrue); ps_step.append(pred)
        else:
            pred = int(torch.argmax(lg["stage_logits"], dim=-1).item())
            ys_stage.append(ytrue); ps_stage.append(pred)

# --- names ---
step_names  = label_list["step_classification"]
stage_names = label_list["stage_classification"]

# --- reports ---
print("\n===== TEST: Step (13-way) class-wise F1 =====")
print(classification_report(
    ys_step, ps_step,
    target_names=step_names,
    digits=4,
    zero_division=0
))

print("\n===== TEST: Stage (3-way) class-wise F1 =====")
print(classification_report(
    ys_stage, ps_stage,
    target_names=stage_names,
    digits=4,
    zero_division=0
))

# Optional: confusion matrices (indices correspond to label_list order)
print("\nConfusion Matrix (Step):")
print(confusion_matrix(ys_step, ps_step))

print("\nConfusion Matrix (Stage):")
print(confusion_matrix(ys_stage, ps_stage))


predict(TEST): 100%|██████████| 224/224 [01:48<00:00,  2.06it/s]


===== TEST: Step (13-way) class-wise F1 =====
              precision    recall  f1-score   support

    step_001     0.8571    0.7500    0.8000         8
    step_002     0.6000    0.6667    0.6316         9
    step_003     0.8000    0.8889    0.8421         9
    step_004     1.0000    1.0000    1.0000         9
    step_005     0.9000    1.0000    0.9474         9
    step_006     0.8571    0.7500    0.8000         8
    step_007     0.6364    0.8750    0.7368         8
    step_008     0.7143    0.5556    0.6250         9
    step_009     0.7000    0.7778    0.7368         9
    step_010     0.7143    0.5556    0.6250         9
    step_011     0.6667    0.8889    0.7619         9
    step_012     1.0000    0.7143    0.8333         7
    step_013     1.0000    0.7778    0.8750         9

    accuracy                         0.7857       112
   macro avg     0.8035    0.7846    0.7858       112
weighted avg     0.8006    0.7857    0.7851       112


===== TEST: Stage (3-way) class